# comets Demo:spike lfp simulation → CoMET vs PCA/UMAP/CEBRA


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from pathlib import Path
import sys

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    code_dir = p / "code"
    if (code_dir / "comets").exists():
        if str(code_dir) not in sys.path:
            sys.path.insert(0, str(code_dir))
        break

import comets
from comets.encoders import Conv1DEncoder, Conv1DEncoderConfig, SkippedConvEncoder, SkippedConvEncoderConfig, CEBRASKipEncoder
from comets.configs import ModalityConfig, SamplingConfig, TrainingConfig
from comets.heads import TwoLayerMLP, LinearHead, IdentityHead, OneLayerMLP


import umap
import cebra
import torch

# -------------------------------------------------------------
# Hippocampal-style simulation for CoMET demo
# -------------------------------------------------------------


from scipy.signal import butter, filtfilt, hilbert
from scipy.ndimage import gaussian_filter1d

In [ ]:


fs = 1000
T_sec = 10
t = np.arange(0, T_sec, 1/fs)
N = t.size
rng = np.random.default_rng(42)

# -------------------------------------------------------------
# 1. Latent trajectory (position + velocity)
# -------------------------------------------------------------
speed = 0.20 + 0.06*np.sin(2*np.pi*0.010*t + 0.7) + 0.03*np.sin(2*np.pi*0.045*t + 1.1)
speed = np.clip(speed, 0.05, None)
x = np.sin(2*np.pi*np.cumsum(speed)/fs)
v = np.gradient(x) * fs

# -------------------------------------------------------------
# 2. Multichannel LFP with realistic theta phase
# -------------------------------------------------------------
def bandpass(sig, fs, lo, hi, order=3):
    b, a = butter(order, [lo, hi], btype="bandpass", fs=fs)
    return filtfilt(b, a, sig)

def pink_noise_like(rng, shape, scale=1.0):
    w = rng.normal(0, 1, size=shape).astype(np.float32)
    p = np.cumsum(w, axis=0)
    p = (p - p.mean(axis=0)) / (p.std(axis=0) + 1e-6)
    return scale * p

# Slight frequency + amplitude variability
f0 = 8.0
f_theta_t = f0 + 0.6*np.sin(2*np.pi*0.02*t) + 0.3*np.sin(2*np.pi*0.007*t + 0.9)
phase_theta = 2*np.pi*np.cumsum(f_theta_t)/fs
env = 0.6 + 0.25*(np.abs(v)/np.max(np.abs(v))+1e-9) + 0.15*np.sin(2*np.pi*0.015*t + 0.3)
env = env.clip(0.2, 1.2)

n_lfp = 12
ch_phase = rng.uniform(-0.7, 0.7, size=n_lfp) * np.pi
Phi = np.stack([np.sin(np.pi*x), np.cos(np.pi*x),
                np.sin(2*np.pi*x), np.cos(2*np.pi*x)], 1)
W_phi = rng.normal(0, 0.35, size=(4, n_lfp))
phi_wiggle = 0.12 * (Phi @ W_phi)
lfp_theta = env[:,None] * np.sin(phase_theta[:,None] + ch_phase[None,:] + phi_wiggle)

# noise + drift
def pink(shape): return pink_noise_like(rng, shape, 0.30)
pinkN  = pink((N, n_lfp))
whiteN = rng.normal(0, 1, size=(N, n_lfp)).astype(np.float32) * 0.22
drift  = np.sin(2*np.pi*0.018*t)[:,None] * rng.normal(0.5,0.2,(1,n_lfp))
lfp = lfp_theta + pinkN + whiteN + drift

# robust theta phase
ref = np.median(lfp, axis=1)
ref_bp = bandpass(ref, fs, 6.0, 10.0)
theta_phase = np.angle(hilbert(ref_bp))


In [ ]:

# -------------------------------------------------------------
# 3. Spiking population (4 classes)
# -------------------------------------------------------------
n_place, n_theta, n_weak, n_noise = 40, 0, 0, 0
n_cell = n_place + n_theta + n_weak + n_noise
spikes = np.zeros((n_cell, N), dtype=bool)

allowed_segments = [(-1.0, -0.55), (-0.15, 0.15), (0.45, 0.8)]
def allowed_mask(pos):
    m = np.zeros_like(pos, dtype=float)
    for a,b in allowed_segments: m += ((pos>=a)&(pos<=b)).astype(float)
    return np.clip(m,0,1)
M_allow = allowed_mask(x)

def sample_spikes_from_rate(rate_hz, refractory_bins=3):
    p = np.clip(rate_hz/fs, 0.0, 0.8)
    s = rng.random(N) < p
    keep, last = [], -10
    for k in np.flatnonzero(s):
        if k - last > refractory_bins:
            keep.append(k); last = k
    out = np.zeros(N,bool); out[keep] = True
    return out

# ---- Place cells ----
centers, widths = [], []
for i in range(n_place):
    a,b = allowed_segments[rng.integers(len(allowed_segments))]
    mu = rng.uniform(a,b); w = rng.uniform(0.08,0.18)
    centers.append(mu); widths.append(w)
    field = np.exp(-0.5*((x - mu)/w)**2)*M_allow
    phi_pref = (np.pi/2) - np.pi*((x - mu)/w)
    phi_pref = (phi_pref + np.pi)%(2*np.pi)-np.pi
    phase_gain = np.exp(6*np.cos(theta_phase - phi_pref))
    lam = 0.05 + 9.0 * field * phase_gain
    spikes[i] = sample_spikes_from_rate(lam)

# ---- Theta cells (lower rate, jitter, skipping) ----
offset = n_place
cycle_idx = (np.arange(N) % int(fs/8)) == 0
for j in range(n_theta):
    pref = rng.uniform(-np.pi, np.pi)
    jitter = np.cumsum(rng.normal(0,0.02,size=N))
    jitter = gaussian_filter1d(jitter, sigma=20)
    inst_pref = (pref + 0.6*jitter)
    speed_gain = 0.5 + 0.5*(np.abs(v)/(np.max(np.abs(v))+1e-9))
    active_cycles = rng.random(np.count_nonzero(cycle_idx)) < 0.6
    cyc_positions = np.flatnonzero(cycle_idx)
    cyc_mask = np.zeros(N)
    for k,on in zip(cyc_positions, active_cycles):
        if on: cyc_mask[k:min(k+int(fs/8),N)] = 1.0
    phase_gain = np.exp(3.5*np.cos(theta_phase - inst_pref))
    lam = (0.3 + 2.2*phase_gain) * speed_gain * cyc_mask
    spikes[offset+j] = sample_spikes_from_rate(lam)

# ---- Weak / multi-peak / directional ----
off2 = offset + n_theta
for j in range(n_weak):
    mu1, mu2 = rng.uniform(-0.9, 0.9, 2)
    w1, w2 = rng.uniform(0.18, 0.35, 2)
    field = np.exp(-0.5*((x - mu1)/w1)**2) + 0.7*np.exp(-0.5*((x - mu2)/w2)**2)
    dir_gain = np.clip(v,0,None)/(np.max(np.abs(v))+1e-9)
    lam = 0.12 + 3.6*field*(1+0.5*dir_gain)
    spikes[off2+j] = sample_spikes_from_rate(lam)

# ---- Noise cells ----
off3 = off2 + n_weak
for j in range(n_noise):
    rw = np.cumsum(rng.normal(0,0.02,size=N))
    lam = np.clip(0.4 + rw, 0.05, 1.8)
    if rng.random() < 0.25:
        b0 = rng.integers(0,N-300); dur = rng.integers(120,300)
        lam[b0:b0+dur] = np.minimum(lam[b0:b0+dur]+rng.uniform(0.0,0.5),1.8)
    spikes[off3+j] = sample_spikes_from_rate(lam)

# smooth rates
rates = gaussian_filter1d(spikes.astype(float), sigma=5, axis=1)


In [ ]:

# -------------------------------------------------------------
# 4. Final modalities for CoMET
# -------------------------------------------------------------
A = lfp.astype(np.float32)   # multichannel LFP
B = rates.T.astype(np.float32) # smoothed firing rates

print("modality_A (LFP):", A.shape)
print("modality_B (spikes):", B.shape)


In [ ]:
# modality configs
modA = ModalityConfig(
    encoder_cls=CEBRASKipEncoder,
    encoder_cfg=SkippedConvEncoderConfig(in_channels=A.shape[1], emb_dim=32),
    win_len=6
)
modB = ModalityConfig(
    encoder_cls=CEBRASKipEncoder,
    encoder_cfg=SkippedConvEncoderConfig(in_channels=B.shape[1], emb_dim=32),
    win_len=6
)

sampling = SamplingConfig(
    batch_size=256,
    pos_offset=3,
    # negative_mask_dist=10   # e.g., mask negatives within ±10 timepoints
)

train_cfg = TrainingConfig(epochs=500,lr=1e-3,temperature= 0.1,proj_dim=8,
    alpha_cross=0.5,  # weight cross vs within
    beta=0.5,         # balance A vs B (default equal)
    head_cls=TwoLayerMLP)

In [ ]:
model = comets.train(A, modA, B, modB, sampling, train_cfg)

In [ ]:

ZA = model.embedA(A)
ZB = model.embedB(B)
print("ZA:", ZA.shape, "ZB:", ZB.shape)

In [ ]:
# 5) Visualize embeddings with our built-in viz
comets.plotemb2(
    ZA, ZB,
    color=x,               # color points by latent angle
    titles=("Modality A", "Modality B"),
    mode="direct",                 # project to 3D PCA for plotting
    dims=[2,3,5]
)

In [ ]:
# 5) Visualize embeddings with our built-in viz
comets.plotemb2(
    ZA, ZB,
    color=x,               # color points by latent angle
    titles=("Modality A", "Modality B"),
    mode="direct"                  # project to 3D PCA for plotting
)

In [ ]:
# 3) Baselines: PCA, UMAP (on each view separately)
# PCA and UMAP baselines
Z1_pca = PCA(n_components=3).fit_transform(A)
Z2_pca = PCA(n_components=3).fit_transform(B)


reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=0)
Z1_umap = reducer.fit_transform(A)
Z2_umap = reducer.fit_transform(B)


In [ ]:
# Optional: CEBRA baseline
cebra_temp = 1

modelA_cebra = cebra.CEBRA(model_architecture='offset10-model',max_iterations=500,
                            batch_size=256, temperature=cebra_temp, verbose = True)
ZA_cebra = modelA_cebra.fit_transform(A)

modelB_cebra = cebra.CEBRA(model_architecture='offset10-model',max_iterations=500,
                            batch_size=256, temperature=cebra_temp, verbose = True)
ZB_cebra = modelB_cebra.fit_transform(B)

In [ ]:
from sklearn.decomposition import PCA


# assume you already have:
# ZA, ZB (CoMET)
# ZA_cebra, ZB_cebra (CEBRA)
# Z1_pca, Z2_pca (PCA)
# Z1_umap, Z2_umap (UMAP)
# theta1 (color variable)

def reduce3(Z):
    """Return a 3-D PCA projection regardless of input dim."""
    Z = np.asarray(Z)
    if Z.shape[1] > 3:
        Z = PCA(n_components=3).fit_transform(Z)
    elif Z.shape[1] < 3:
        pad = np.zeros((Z.shape[0], 3 - Z.shape[1]))
        Z = np.concatenate([Z, pad], axis=1)
    return Z[:, 0], Z[:, 1], Z[:, 2]

# ------------------------------------------
# Create figure: 2 rows x 4 columns
# ------------------------------------------
fig = plt.figure(figsize=(18, 8))

titles_top = ["CoMET A (PCA-reduced)", "PCA A", "UMAP A", "CEBRA A (PCA-reduced)"]
titles_bot = ["CoMET B (PCA-reduced)", "PCA B", "UMAP B", "CEBRA B (PCA-reduced)"]

# Apply PCA to CoMET / CEBRA before plotting
ZA_red = PCA(n_components=3).fit_transform(ZA)
ZB_red = PCA(n_components=3).fit_transform(ZB)
ZA_cebra_red = PCA(n_components=3).fit_transform(ZA_cebra)
ZB_cebra_red = PCA(n_components=3).fit_transform(ZB_cebra)

datasets_top = [ZA_red, Z1_pca, Z1_umap, ZA_cebra_red]
datasets_bot = [ZB_red, Z2_pca, Z2_umap, ZB_cebra_red]

for i in range(4):
    # ---- Top row (view A) ----
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    X, Y, Z = reduce3(datasets_top[i])
    ax.scatter(X, Y, Z, c=x, cmap='viridis', s=5)
    ax.set_title(titles_top[i], fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

    # ---- Bottom row (view B) ----
    ax = fig.add_subplot(2, 4, i + 5, projection='3d')
    X, Y, Z = reduce3(datasets_bot[i])
    ax.scatter(X, Y, Z, c=x, cmap='viridis', s=5)
    ax.set_title(titles_bot[i], fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

fig.suptitle("Embedding Comparison (colored by theta1)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 6) Decoding: predict latent angles from embeddings
# We'll use Ridge regression to predict sin/cos of angle for circular decoding,
# then compute mean R^2 on sin and cos.

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

In [ ]:

def linear_decode_score(Z, y):
    """Decode continuous variable y from embedding Z using ridge regression."""
    Z = np.asarray(Z)
    y = np.asarray(y).ravel()
    n = len(y)
    ntr = int(0.7 * n)
    Xtr, Xte = Z[:ntr], Z[ntr:]
    ytr, yte = y[:ntr], y[ntr:]
    model = Ridge(alpha=1.0)
    model.fit(Xtr, ytr)
    yhat = model.predict(Xte)
    return r2_score(yte, yhat)

# -----------------------------
# Example decoding comparison
# -----------------------------
scores = {}
toval = x  # latent true position

scores['CoMETs_A'] = linear_decode_score(ZA, toval)
scores['CoMETs_B'] = linear_decode_score(ZB, toval)
scores['PCA_A']    = linear_decode_score(Z1_pca, toval)
scores['PCA_B']    = linear_decode_score(Z2_pca, toval)
scores['UMAP_A']   = linear_decode_score(Z1_umap, toval)
scores['UMAP_B']   = linear_decode_score(Z2_umap, toval)
scores['CEBRA_A']  = linear_decode_score(ZA_cebra, toval)
scores['CEBRA_B']  = linear_decode_score(ZB_cebra, toval)

print("Decoding scores (R² of position x):")
for k, v in scores.items():
    print(f"{k:10s}: {v:.3f}")